In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import iplotx as ipx
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import numpy.typing as npt
import polars as pl
import scipy as sp
import seaborn as sns
from ising.model import FitMethod, UpdateMethod

from climate_attitudes.dataset import Dataset
from climate_attitudes.datasets.reduced_no_imputation import schema
from climate_attitudes.settings import Config

# from climate_attitudes.datasets.small_four_waves import schema
from climate_attitudes.visualisation import DIVERGING_CMAP, configure_mpl
from ising import Ising

configure_mpl(Path("../fonts/"))

np.set_printoptions(linewidth=200)

RANDOM_SEED = 202607101941

DATA_PATH = Path("../reports/thesis/results/data/model/all_interventions/")
schema = schema.post_index()

rng = np.random.default_rng(RANDOM_SEED)

labels = schema.get_short_names("measurement")

In [ ]:
data = np.load(
    Path("../reports/thesis/results/data/model/bootstrapped_fit/ising_no_structure.npz")
)

In [ ]:
interactions = data["params"][:, 8:]

In [ ]:
mean = interactions.mean(axis=0)
lo, hi = np.percentile(interactions, (2.5, 97.5), axis=0)

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 2.5), constrained_layout=True)

sort_idx = np.argsort(mean)
mean = mean[sort_idx]
lo = lo[sort_idx]
hi = hi[sort_idx]

ax.plot(mean, np.arange(mean.size), "o", markersize=0.5, zorder=4, label="Mean effect")
marker, _, bar = ax.errorbar(
    mean,
    np.arange(mean.size),
    xerr=[mean - lo, hi - mean],
    linewidth=0,
    ecolor="tab:blue",
    zorder=3,
    label="95% CI",
)
plt.setp(bar[0], capstyle="round")
marker.set_fillstyle("none")
bar[0].set_linewidth(1.5)
bar[0].set_alpha(0.3)

ax.spines.top.set_visible(False)
ax.spines.right.set_visible(False)
ax.spines.left.set_visible(False)
# ax.yaxis.set_visible(False)
ax.set_yticks([])
ax.set_ylim(-1, None)
ax.set_xlabel(r"Interaction effect ($J_{i,j}$)")
ax.set_ylabel("Edge")

ax.legend(
    loc="lower center",
    bbox_to_anchor=(0.5, 1.05),
    ncols=2,
    frameon=False,
)
fig.savefig("edge_accuracy.pdf", bbox_inches="tight", dpi=300)

Edge selection frequency

In [ ]:
nonzero_interactions = interactions.copy()
nonzero_interactions[abs(nonzero_interactions) < 1e-2] = 0

p_selected = (
    (~np.isclose(nonzero_interactions, 0)).sum(axis=0) / interactions.shape[0]
).reshape((8, 8))

In [ ]:
G = nx.from_numpy_array(p_selected < 1.0, create_using=nx.DiGraph)
for i, j in G.edges():
    G[i][j]["p"] = p_selected[i, j]

for node, label in enumerate(schema.get_short_names("measurement")):
    G.nodes[node]["label"] = label

In [ ]:
ipx_style: dict[str, dict[str, Any]] | list[str | dict[str, dict[str, Any]]] = [
    "hollow",
    {
        "vertex": {
            "size": "label",
            "edgecolor": "black",
            "linewidth": 0.5,
            "facecolor": "#FFFFFF",
            # "label": dict(hpadding=23, vpadding=30, color="black", size=8),
            "label": dict(color="black", size=6, hpadding=35),
            "zorder": 7,
            "alpha": 1.0,
        },
        "edge": {
            "alpha": 1,
            "color": "#BBBBBB",
            "zorder": 1,
            "shrink": 7,
            "arrow": {
                "width": 3,
            },
        },
    },
]

fig, ax = plt.subplots(constrained_layout=True)

edge_linewidths = {(u, v): 1 for u, v, z in G.edges(data=True)}
vertex_labels = [z["label"] for _, z in G.nodes(data=True)]
ipx_style[1]["edge"]["color"] = [z["p"] for *_, z in G.edges(data=True)]
ipx_style[1]["edge"]["cmap"] = DIVERGING_CMAP
ipx_style[1]["edge"]["norm"] = mcolors.Normalize(vmin=-1.0, vmax=1.0)


network_artist = ipx.network(
    G,
    layout=nx.circular_layout(G),
    vertex_labels=vertex_labels,
    edge_curved=True,
    edge_linewidth=edge_linewidths,
    # vertex_labels=True,
    margin=0.1,
    style=ipx_style,
    ax=ax,
)[0]

In [ ]:
p_selected

In [ ]:
schema.get_short_names("measurement")

In [ ]:
real_params = np.load(
    Path("../reports/thesis/results/data/model/fit_full_asym_ising_no_structure.npz")
)["params"][8:].reshape((8, 8))
real_params[abs(real_params) < 1e-2] = 0

In [ ]:
real_params

### Cross-validation relative entropy

5-fold cross-validation; calculate relative entropy between model and observations. 

For a given individual and spin, the relative entropy is:

$$- \sum_{s \in \pm 1} P[\operatorname{Bin}(x_i^{(m)}) = s] \cdot \log P[S_i^{t+1} = s | \boldsymbol{S}^{t} = \boldsymbol{s}^t] - H(\operatorname{Bin}(X_i^{(m)}))$$

First define a function to calculate the entropy of an array under the binarisation process

In [ ]:
def binarisation_entropy(p: npt.NDArray[np.float64]) -> npt.NDArray[np.float64]:
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

In [ ]:
p = np.linspace(0.0001, 0.9999, 1_000)
bin_ent = binarisation_entropy(p)
fig, ax = plt.subplots(figsize=(2.5, 3), constrained_layout=True)
ax.set_aspect("equal")

ax.plot(p, bin_ent, clip_on=False)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.spines.top.set_visible(False)
ax.spines.right.set_visible(False)
ax.set_xlabel(r"$P[\operatorname{Bin}(x) = +1]$")
ax.set_ylabel("Entropy");

Next we want to calculate the expected surprise when sampling values using the model, but observing the true binarised values.

First we define a few helper functions to calculate the probability of sampling $+1$ for each individual in a dataset, given their previous observed state and a calibrated model. Note that we must account for the uncertainty in the binarisation of the initial state here as well. In practice this means that for each individual and possible configuration $\boldsymbol{s}$, we must calculate the probability that their initial state is binarised to $\boldsymbol{s}$, and use the law of total probability to calculate the probability of sampling $+1$ for each of the spins in the following state.

In [ ]:
def activation_prob_given_binary_prev(
    prev: npt.NDArray[np.int64], model: Ising
) -> npt.NDArray[np.float64]:
    X = np.ones(prev.shape[0])
    heff = model.parallel_glauber_theta_batch(prev, X, model.h, model.j, model.adj)
    p = np.exp(heff - np.log(2 * np.cosh(heff)))
    return p


def activation_prob_given_prev_binarisation_p(
    prev_p: npt.NDArray[np.float64], model: Ising
) -> npt.NDArray[np.float64]:
    # Create array where rows are all possible model configurations
    N = model.size
    S = (2 * ((np.arange(1 << N)[:, None] >> np.arange(N)) & 1) - 1).astype(np.float64)

    # Calculate probability that binarisation yields each configuration
    bin_p = np.empty((prev_p.shape[0], 2**N), dtype=np.float64)
    for m in range(prev_p.shape[0]):
        bin_p[m] = np.exp(
            np.log((1 - S) / 2 * (1 - prev_p[m]) + (1 + S) / 2 * prev_p[m]).sum(axis=-1)
        )

    # Calculate activation probability of next state given each initial binarisation
    conditional_next_activation_prob = activation_prob_given_binary_prev(S, model)

    # For each individual, calculate total activation probability of next state
    next_activation_prob = np.exp(
        np.log(bin_p)[..., None] + np.log(conditional_next_activation_prob)
    ).sum(axis=1)
    return next_activation_prob

Now we can define the expected surprise:

In [ ]:
def expected_excess_sampling_surprise(
    bin_p: npt.NDArray[np.float64],
    model: Ising,
) -> npt.NDArray[np.float64]:
    if bin_p.shape[1] != 2:
        raise ValueError(f"Expected bin_p with two data points, found {bin_p.shape[1]}")
    prev_bin_p, next_bin_p = np.swapaxes(bin_p, 0, 1)

    conditional_activation_prob = activation_prob_given_prev_binarisation_p(
        prev_bin_p,
        model,
    )

    # Calculate expected surprise when expecting model samples but observing data
    cross_entropy = -(1 - next_bin_p) * np.log2(
        1 - conditional_activation_prob
    ) - next_bin_p * np.log2(conditional_activation_prob)

    # Calculate the expected _excess_ surprise by subtracting binarisation entropy
    next_obs_entropy = binarisation_entropy(next_bin_p)
    expected_excess_surprise = cross_entropy - next_obs_entropy
    return expected_excess_surprise

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(
    config,
    name="reduced_no_imputation",
    with_imputation=False,
    verbose=False,
)

sigma_path = Path("../reports/thesis/results/data/methods/binarisation_sigma.json")
lambda_path = Path(
    "../reports/thesis/results/data/model_fit/optimised_regularisation.json"
)

with sigma_path.open("r") as f:
    sigma = json.load(f)["sigma"]

with lambda_path.open("r") as f:
    reg_results = json.load(f)
    λ_asym = reg_results["ising"]["full"]
    λ_sym = reg_results["sym_ising"]["full"]

# Fit model
Y, _, P, *_ = dataset.indices_to_numpy(
    kind="time-series",
    binarise=True,
    scale=sigma,
    seed=rng,
    binarisation_dist="gaussian",
)
model = Ising.fit(
    y=P,
    optim_method=FitMethod.TIME_SERIES,
    update_method=UpdateMethod.SYNCHRONOUS,
    rng=rng.spawn(1)[0],
    adj=None,
    self_loops=True,
    w=λ_asym,
    node_labels=labels,
)
null_model = Ising.fit(
    y=P,
    optim_method=FitMethod.TIME_SERIES,
    update_method=UpdateMethod.SYNCHRONOUS,
    rng=rng.spawn(1)[0],
    adj=np.eye(8, dtype=np.bool),
    self_loops=True,
    w=λ_asym,
    node_labels=labels,
)

In [ ]:
np.round(
    (
        expected_excess_sampling_surprise(P[1077:1078], model)
        - expected_excess_sampling_surprise(P[1077:1078], null_model)
    ),
    2,
)

Cross-validate

In [ ]:
# https://stackoverflow.com/questions/42592584/can-i-use-a-numpy-array-to-generate-folds-for-cross-validation
# P = P[:100]
M = P.shape[0]
K = 5
eval_idx_sets = np.array_split(rng.choice(np.arange(M), size=M, replace=False), K)

kl_results_train = []
kl_results_eval = []

for k in range(K):
    mask_eval = np.ones(M, dtype=bool)
    mask_eval[eval_idx_sets[k]] = False
    P_train = P[mask_eval]
    P_eval = P[eval_idx_sets[k]]

    cv_model = Ising.fit(
        y=P_train,
        optim_method=FitMethod.TIME_SERIES,
        update_method=UpdateMethod.SYNCHRONOUS,
        rng=rng.spawn(1)[0],
        adj=None,
        self_loops=True,
        w=λ_asym,
    )
    kl_results_train.append(expected_excess_sampling_surprise(P_train, cv_model))
    kl_results_eval.append(expected_excess_sampling_surprise(P_eval, cv_model))

In [ ]:
fig, ax = plt.subplots(figsize=(3, 1.5), constrained_layout=True)

for i, res in enumerate(kl_results_train):
    sns.kdeplot(
        res.mean(axis=1),
        ax=ax,
        clip=(0, None),
        color="tab:blue",
        linewidth=0.5,
        label="Calibration" if i == 0 else None,
    )

for i, res in enumerate(kl_results_eval):
    sns.kdeplot(
        res.mean(axis=1),
        ax=ax,
        clip=(0, None),
        color="tab:orange",
        linewidth=0.5,
        label="Holdout" if i == 0 else None,
    )

ax.set_xlim(0, None)
ax.spines.top.set_visible(False)
ax.spines.right.set_visible(False)

ax.legend(
    loc="lower center",
    bbox_to_anchor=(0.5, 1.0),
    ncols=2,
    frameon=False,
)

fig.savefig("cross_validation_relative_entropy.png", bbox_inches="tight")

In [ ]:
# https://stackoverflow.com/questions/42592584/can-i-use-a-numpy-array-to-generate-folds-for-cross-validation
# P = P[:100]
M = P.shape[0]
K = 10
eval_idx_sets = np.array_split(rng.choice(np.arange(M), size=M, replace=False), K)

kl_diffs_eval = []
kl_diffs_calibration = []

kl_eval = []
kl_calibration = []
kl_eval_null = []
kl_calibration_null = []

for k in range(K):
    mask_eval = np.ones(M, dtype=bool)
    mask_eval[eval_idx_sets[k]] = False
    P_train = P[mask_eval]
    P_eval = P[eval_idx_sets[k]]

    cv_model = Ising.fit(
        y=P_train,
        optim_method=FitMethod.TIME_SERIES,
        update_method=UpdateMethod.SYNCHRONOUS,
        rng=rng.spawn(1)[0],
        adj=None,
        self_loops=True,
        w=λ_asym,
    )
    cv_model_null = Ising.fit(
        y=P_train,
        optim_method=FitMethod.TIME_SERIES,
        update_method=UpdateMethod.SYNCHRONOUS,
        rng=rng.spawn(1)[0],
        adj=np.eye(8, dtype=np.bool),
        self_loops=True,
        w=λ_asym,
    )
    kl_diffs_calibration.append(
        expected_excess_sampling_surprise(P_train, cv_model).mean(axis=-1)
        - expected_excess_sampling_surprise(P_train, cv_model_null).mean(axis=-1)
    )
    kl_diffs_eval.append(
        expected_excess_sampling_surprise(P_eval, cv_model).mean(axis=-1)
        - expected_excess_sampling_surprise(P_eval, cv_model_null).mean(axis=-1)
    )
    kl_eval.append(expected_excess_sampling_surprise(P_eval, cv_model).mean(axis=-1))
    kl_eval_null.append(
        expected_excess_sampling_surprise(P_eval, cv_model_null).mean(axis=-1)
    )
    kl_calibration.append(
        expected_excess_sampling_surprise(P_train, cv_model).mean(axis=-1)
    )
    kl_calibration_null.append(
        expected_excess_sampling_surprise(P_train, cv_model_null).mean(axis=-1)
    )

In [ ]:
eval_idxes_all = np.concat(eval_idx_sets)
sort_order = np.argsort(eval_idxes_all)

In [ ]:
kl_eval_all = np.concat(kl_eval)[sort_order]
kl_eval_null_all = np.concat(kl_eval_null)[sort_order]
kl_diffs_eval_all = np.concat(kl_diffs_eval)[sort_order]

In [ ]:
lo, hi = np.percentile(kl_diffs_eval_all, (1, 99))

In [ ]:
sample_idxes_lo = rng.choice(
    np.arange(M)[kl_diffs_eval_all <= lo], size=4, replace=False
)
sample_idxes_hi = rng.choice(
    np.arange(M)[kl_diffs_eval_all >= hi], size=4, replace=False
)

In [ ]:
sample_idxes_hi

In [ ]:
hi

In [ ]:
kl_eval_all[385]

In [ ]:
kl_eval_null_all[385]

In [ ]:
np.argwhere(eval_idxes_all == 594)

In [ ]:
kl_diffs_eval_all[267]

In [ ]:
fig, axes = plt.subplots(
    ncols=2, figsize=(5, 1.5), constrained_layout=True, sharex=True
)


for i, res in enumerate(kl_diffs_eval):
    sns.kdeplot(
        res,
        ax=axes[0],
        clip=(None, None),
        color="tab:orange",
        linewidth=0.5,
        label="Holdout" if i == 0 else None,
    )

for i, res in enumerate(kl_diffs_eval):
    sns.ecdfplot(
        res,
        ax=axes[1],
        color="tab:orange",
        linewidth=0.5,
        label="Holdout" if i == 0 else None,
    )

# ax.set_xlim(0, None)
for ax in axes:
    ax.spines.top.set_visible(False)
    ax.spines.right.set_visible(False)

axes[0].set_title("Probability Density (KDE)")
axes[1].set_title("ECDF")

fig.savefig("cross_validation_relative_entropy.png", bbox_inches="tight")

In [ ]:
N = P.shape[-1]

fig, axes = plt.subplots(
    ncols=4,
    nrows=2,
    figsize=(5.77, 3),
    constrained_layout=True,
    sharex=True,
    sharey=True,
)

for ax, idx in zip(axes[0].flatten(), sample_idxes_lo, strict=True):
    ax.plot(
        np.arange(N),
        P[idx][0],
        "o",
        markersize=3,
        clip_on=False,
        markerfacecolor="none",
        color="k",
        markeredgewidth=0.6,
    )

    for i in range(N):
        if np.abs(P[idx, 0, i] - P[idx, 1, i]) < 0.1:
            continue
        color = "tab:blue" if P[idx, 0, i] < P[idx, 1, i] else "tab:orange"
        ax.annotate(
            "",
            xytext=(i, P[idx][0][i]),
            xy=(i, P[idx][1][i]),
            arrowprops=dict(arrowstyle="->", linewidth=0.6, shrinkB=0, color=color),
        )

    ax.set_ylim(-0.05, 1.05)

    ax.spines.top.set_visible(False)
    ax.spines.right.set_visible(False)

for ax, idx in zip(axes[1].flatten(), sample_idxes_hi, strict=True):
    ax.plot(
        np.arange(N),
        P[idx][0],
        "o",
        markersize=3,
        clip_on=False,
        markerfacecolor="none",
        color="k",
        markeredgewidth=0.6,
    )

    for i in range(N):
        if np.abs(P[idx, 0, i] - P[idx, 1, i]) < 0.1:
            continue
        color = "tab:blue" if P[idx, 0, i] < P[idx, 1, i] else "tab:orange"
        ax.annotate(
            "",
            xytext=(i, P[idx][0][i]),
            xy=(i, P[idx][1][i]),
            arrowprops=dict(arrowstyle="->", linewidth=0.6, shrinkB=0, color=color),
        )

    ax.set_ylim(-0.05, 1.05)

    ax.spines.top.set_visible(False)
    ax.spines.right.set_visible(False)

fig.text(
    1.02,
    0.85,
    r"$\cal{M} \succ \cal{M}_\text{null}$",
    rotation=0,
    va="center",
    ha="left",
    fontsize=10,
)

fig.text(
    1.02,
    0.5,
    r"$\cal{M} \prec \cal{M}_\text{null}$",
    rotation=0,
    va="center",
    ha="left",
    fontsize=10,
)

fig.supylabel("P$[S_i = +1]$", y=0.7)
for ax in axes[1]:
    ax.set_xticks(
        np.arange(8), schema.get_short_names("measurement"), rotation=90, fontsize=9
    )

# fig.savefig("high_error_participants.png", bbox_inches="tight")

In [ ]:
lo, hi

In [ ]:
N = P.shape[-1]

fig, axes = plt.subplots(
    ncols=4,
    nrows=2,
    figsize=(5.77, 3),
    constrained_layout=True,
    sharex=True,
    sharey=True,
)

all_kl_results_eval = np.concat(kl_results_eval)
all_eval_idxes = np.concat(eval_idx_sets)
most_weird = np.argsort(all_kl_results_eval.mean(axis=-1))[-8:]
for ax, cv_idx in zip(axes.flatten(), most_weird, strict=True):
    # fig, ax = plt.subplots(figsize=(3, 2.15), constrained_layout=True)

    idx = all_eval_idxes[cv_idx]
    ax.plot(
        np.arange(N),
        P[idx][0],
        "o",
        markersize=3,
        clip_on=False,
        markerfacecolor="none",
        color="k",
        markeredgewidth=0.6,
    )

    for i in range(N):
        if np.abs(P[idx, 0, i] - P[idx, 1, i]) < 0.1:
            continue
        color = "tab:blue" if P[idx, 0, i] < P[idx, 1, i] else "tab:orange"
        ax.annotate(
            "",
            xytext=(i, P[idx][0][i]),
            xy=(i, P[idx][1][i]),
            arrowprops=dict(arrowstyle="->", linewidth=0.6, shrinkB=0, color=color),
        )

    ax.set_ylim(-0.05, 1.05)

    ax.spines.top.set_visible(False)
    ax.spines.right.set_visible(False)

fig.supylabel("P$[S_i = +1]$", y=0.7)
for ax in axes[1]:
    ax.set_xticks(
        np.arange(8), schema.get_short_names("measurement"), rotation=90, fontsize=9
    )

fig.savefig("high_error_participants.png", bbox_inches="tight")

In [ ]:
idx = 149

most_weird = np.argsort(kl_results_eval[0].mean(axis=-1))[-10:]
for cv_idx in most_weird:
    idx = eval_idx_sets[0][cv_idx]
    fig, ax = plt.subplots(figsize=(4.5, 3), constrained_layout=True)
    sns.barplot(
        pl.DataFrame(
            {
                "Spin": schema.get_short_names("measurement"),
                "Wave 3": np.round(2 * P[idx][0] - 1, decimals=2),
                "Wave 4": np.round(2 * P[idx][1] - 1, decimals=2),
            }
        ).unpivot(
            index="Spin", variable_name="Wave", value_name="Activation probability"
        ),
        x="Spin",
        y="Activation probability",
        hue="Wave",
        width=0.8,
        ax=ax,
    )
    ax.bar_label(ax.containers[0], fontsize=8)
    ax.bar_label(ax.containers[1], fontsize=8)

    ax.set_xticks(
        np.arange(8),
        schema.get_short_names("measurement"),
        rotation=45,
        horizontalalignment="right",
    )
    ax.set_ylim(-1.35, 1.35)

    ax.spines.top.set_visible(False)
    ax.spines.right.set_visible(False)

    ax.set_xlabel("")

    ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.05), ncols=2, frameon=False)

## Revisions based on feedback from Vítor

### Reliability of transitions

**This is actually not straightforward, since the variables are mostly ordinal, so have clear bins for probability. Perhaps we bin according to the ordinal scale.**

For each participant, calculate the probability that their next spin state is 1 (can do a single spin for now, and maybe aggregate later)

In [ ]:
p_activation = activation_prob_given_prev_binarisation_p(P[:, 0], model)

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3), constrained_layout=True)
ax.set_aspect("equal")

ax.plot(p_activation[:, 3], P[:, 1, 3], "o", markersize=1)

Bin the activation probabilities (e.g., into 10 bins).

Plot the bin centers against the observed mean probability of binarisation to $+1$

**Good fits**

- CC Real, CC Human: [-1, 0, 1]
- CC Worry, CC Others Worry, Weather worry: [-1, -0.15, 0.15, 1]

In [ ]:
M = P.shape[0]
K = 5
eval_idx_sets = np.array_split(rng.choice(np.arange(M), size=M, replace=False), K)
buckets = np.linspace(-1, 1, 11)
# buckets = np.array([-1, -0.5, 0, 0.5, 1])

prob_bins_eval = np.zeros((M, P.shape[-1]), dtype=np.float64)
bin_edges = sp.stats.norm.cdf(buckets / sigma)
bin_centers = bin_edges[:-1] + np.diff(bin_edges) / 2
n_bins = bin_edges.size - 1

In [ ]:
for k in range(K):
    mask_eval = np.ones(M, dtype=bool)
    mask_eval[eval_idx_sets[k]] = False
    P_train = P[mask_eval]
    P_eval = P[eval_idx_sets[k]]

    cv_model = Ising.fit(
        y=P_train[:, [0, 1]],
        optim_method=FitMethod.TIME_SERIES,
        update_method=UpdateMethod.SYNCHRONOUS,
        rng=rng.spawn(1)[0],
        adj=None,
        self_loops=True,
        w=λ_asym,
    )

    p_activation = activation_prob_given_prev_binarisation_p(P_eval[:, 0], cv_model)
    prob_bins_eval[eval_idx_sets[k]] = np.argmax(
        (p_activation[:, :, None] < bin_edges[1:]), axis=-1
    )

In [ ]:
# p_activation = activation_prob_given_prev_binarisation_p(P[:, 0], model)
# prob_bins_calibration = np.argmax((p_activation[:,:,None] < bin_edges[1:]), axis=-1)

In [ ]:
mean_binarisation_prob_eval = np.full((8, n_bins), fill_value=np.nan)
# mean_binarisation_prob_calibration = np.full_like(
#     mean_binarisation_prob_eval,
#     fill_value=np.nan,
# )
for spin in range(P.shape[-1]):
    for _bin in range(n_bins):
        in_bin_eval = prob_bins_eval[:, spin] == _bin
        if in_bin_eval.sum() > 10:
            mean_binarisation_prob_eval[spin, _bin] = P[in_bin_eval, 1, spin].mean()


fig, axes = plt.subplots(
    ncols=4,
    nrows=2,
    figsize=(5.77, 3),
    constrained_layout=True,
    sharex=True,
    sharey=True,
)
labels = schema.post_index().get_short_names("measurement")

for i, ax in enumerate(axes.flatten()):
    if i == P.shape[-1]:
        break
    ax.plot(
        bin_centers, mean_binarisation_prob_eval[i], "o-", markersize=2, linewidth=0.75
    )
    ax.plot([0, 1], [0, 1], linestyle="dashed", linewidth=0.35, color="k")

    ax.set_aspect("equal")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    ax.spines.top.set_visible(False)
    ax.spines.right.set_visible(False)
    ax.set_title(labels[i], fontsize=9)

    x = bin_centers[~np.isnan(mean_binarisation_prob_eval[i])]
    y = mean_binarisation_prob_eval[i][~np.isnan(mean_binarisation_prob_eval[i])]

    res = y - x
    ss_res = (res**2).sum()
    ss_tot = ((x - x.mean()) ** 2).sum()
    # print(obs_probs)
    print(f"R2 = {1 - ss_res / ss_tot}")

fig.supxlabel(r"$P(S_i^{t+1} \mid \boldsymbol{S}^{t})$")
fig.supylabel("Mean binarisation probability")

Debug Politics

In [ ]:
mean_binarisation_prob_eval = np.full((8, n_bins), fill_value=np.nan)
# mean_binarisation_prob_calibration = np.full_like(
#     mean_binarisation_prob_eval,
#     fill_value=np.nan,
# )
spin = 5
for _bin in range(n_bins):
    in_bin_eval = prob_bins_eval[:, spin] == _bin
    if in_bin_eval.sum() > 10:
        mean_binarisation_prob_eval[spin, _bin] = P[in_bin_eval, 1, spin].mean()


fig, ax = plt.subplots(figsize=(1.5, 1.5), constrained_layout=True)
labels = schema.post_index().get_short_names("measurement")

ax.plot(
    bin_centers, mean_binarisation_prob_eval[spin], "o-", markersize=2, linewidth=0.75
)
ax.plot([0, 1], [0, 1], linestyle="dashed", linewidth=0.35, color="k")

ax.set_aspect("equal")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.spines.top.set_visible(False)
ax.spines.right.set_visible(False)
ax.set_title(labels[spin], fontsize=9)

fig.supxlabel(r"$P(S_i^{t+1} \mid \boldsymbol{S}^{t})$")
# fig.supylabel("Mean binarisation probability")

In [ ]:
mean_binarisation_prob_eval = np.full((8, n_bins), fill_value=np.nan)
# mean_binarisation_prob_calibration = np.full_like(
#     mean_binarisation_prob_eval,
#     fill_value=np.nan,
# )
for spin in range(P.shape[-1]):
    for _bin in range(n_bins):
        in_bin_eval = prob_bins_eval[:, spin] == _bin
        if in_bin_eval.sum() > 10:
            mean_binarisation_prob_eval[spin, _bin] = P[in_bin_eval, 1].mean()


fig, axes = plt.subplots(
    ncols=4,
    nrows=2,
    figsize=(5.77, 3),
    constrained_layout=True,
    sharex=True,
    sharey=True,
)
labels = schema.post_index().get_short_names("measurement")

for i, ax in enumerate(axes.flatten()):
    if i == P.shape[-1]:
        break
    ax.plot(
        bin_centers, mean_binarisation_prob_eval[i], "o-", markersize=2, linewidth=0.75
    )
    ax.plot([0, 1], [0, 1], linestyle="dashed", linewidth=0.35, color="k")

    ax.set_aspect("equal")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    ax.spines.top.set_visible(False)
    ax.spines.right.set_visible(False)
    ax.set_title(labels[i], fontsize=9)

fig.supxlabel(r"$P(S_i^{t+1} \mid \boldsymbol{S}^{t})$")
fig.supylabel("Mean binarisation probability")

### How often do people change?

In [ ]:
schema.get_short_names("measurement")

In [ ]:
P[4]

In [ ]:
for t in range(1, P.shape[1]):
    X0 = sp.stats.Normal(mu=0, sigma=1).icdf(P[:, 0]) * sigma
    X1 = sp.stats.Normal(mu=0, sigma=1).icdf(P[:, t]) * sigma
    dX = X1 - X0

    labels = schema.post_index().get_short_names("measurement")

    bins = [
        [-1.5, -0.5, 0.5, 1.5],
        # [-2, 0, 2],
        [-4 / 3, -2 / 3, 0, 2 / 3, 4 / 3],
        # [-4 / 3, -2 / 3, 0, 2 / 3, 4 / 3],
        [-4 / 3, -2 / 3, 0, 2 / 3, 4 / 3],
        [-1.25, -0.75, -0.25, 0.25, 0.75, 1.25],
        [-1.25, -0.75, -0.25, 0.25, 0.75, 1.25],
        # [-1.25, -0.75, -0.25, 0.25, 0.75, 1.25],
    ]

    diff_bins = []

    for _bins in bins:
        centers = (np.asarray(_bins[:-1]) + np.asarray(_bins[1:])) / 2
        diffs = np.asarray([centers[-1] - centers[i] for i in range(len(centers))])
        diffs = np.sort(np.concat((diffs, -diffs[:-1])))
        dx = diffs[1] - diffs[0]
        bin_edges = np.linspace(diffs[0] - dx / 2, diffs[-1] + dx / 2, diffs.size + 1)
        diff_bins.append(bin_edges)

    fig, axes = plt.subplots(
        ncols=4,
        nrows=2,
        figsize=(5, 2.3),
        constrained_layout=True,
        sharex=True,
        sharey=True,
    )
    for i, ax in enumerate(axes.flatten()):
        if i == dX.shape[1]:
            break
        sns.histplot(
            dX[:, i],
            ax=ax,
            bins=diff_bins[i],
            stat="probability",
            shrink=0.18 * (len(diff_bins[i]) - 1) / 2,
        )

        ax.set_title(labels[i], fontsize=10)
        ax.spines.top.set_visible(False)
        ax.spines.right.set_visible(False)
        ax.set_ylim(0, 1)
        ax.set_ylabel("")

    fig.supylabel("Empirical Probability", y=0.57)
    fig.supxlabel("Change in observed state")

### Relative-entropy comparison with null model

In [ ]:
# kl_asym = expected_excess_sampling_surprise(P, model).mean(axis=-1)
# kl_asym_null = expected_excess_sampling_surprise(P, null_model).mean(axis=-1)

# kl_sym = expected_excess_sampling_surprise(P, sym_model).mean(axis=-1)
# kl_sym_null = expected_excess_sampling_surprise(P, sym_null_model).mean(axis=-1)

M = P.shape[0]
K = 10
eval_idx_sets = np.array_split(rng.choice(np.arange(M), size=M, replace=False), K)

kl_results_train = []
kl_results_train_null = []
kl_results_eval = []
kl_results_eval_null = []
diffs_train = []
diffs_eval = []

for k in range(K):
    mask_eval = np.ones(M, dtype=bool)
    mask_eval[eval_idx_sets[k]] = False
    P_train = P[mask_eval]
    P_eval = P[eval_idx_sets[k]]

    cv_model = Ising.fit(
        y=P_train,
        optim_method=FitMethod.TIME_SERIES,
        update_method=UpdateMethod.SYNCHRONOUS,
        rng=rng.spawn(1)[0],
        adj=None,
        self_loops=True,
        w=λ_asym,
    )
    cv_model_null = Ising.fit(
        y=P_train,
        optim_method=FitMethod.TIME_SERIES,
        update_method=UpdateMethod.SYNCHRONOUS,
        rng=rng.spawn(1)[0],
        adj=np.eye(P.shape[-1], dtype=np.bool),
        self_loops=True,
        w=λ_asym,
    )

    kl_results_train.append(expected_excess_sampling_surprise(P_train, cv_model).mean())
    kl_results_eval.append(expected_excess_sampling_surprise(P_eval, cv_model).mean())
    kl_results_train_null.append(
        expected_excess_sampling_surprise(P_train, cv_model_null).mean()
    )
    kl_results_eval_null.append(
        expected_excess_sampling_surprise(P_eval, cv_model_null).mean()
    )
    diffs_train.append(
        (
            expected_excess_sampling_surprise(P_train, cv_model)
            - expected_excess_sampling_surprise(P_train, cv_model_null)
        ).mean()
    )
    diffs_eval.append(
        (
            expected_excess_sampling_surprise(P_eval, cv_model)
            - expected_excess_sampling_surprise(P_eval, cv_model_null)
        ).mean()
    )

kl_results_train = np.asarray(kl_results_train)
kl_results_train_null = np.asarray(kl_results_train_null)
kl_results_eval = np.asarray(kl_results_eval)
kl_results_eval_null = np.asarray(kl_results_eval_null)
diffs_train = np.asarray(diffs_train)
diffs_eval = np.asarray(diffs_eval)

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(5.77, 2.25), constrained_layout=True)

plot_data = pl.DataFrame(
    {
        "Relative Entropy": np.concat(
            (
                kl_results_train,
                kl_results_train_null,
                kl_results_eval,
                kl_results_eval_null,
            )
        ),
        "Model": ["Calibration"] * kl_results_train.size * 2
        + ["Validation"] * kl_results_eval.size * 2,
        "Conditions": ["Full connectivity"] * kl_results_train.size
        + ["Self-only"] * kl_results_train_null.size
        + ["Full connectivity"] * kl_results_eval.size
        + ["Self-only"] * kl_results_eval_null.size,
    }
)

sns.barplot(
    plot_data,
    x="Conditions",
    y="Relative Entropy",
    hue="Model",
    order=["Self-only", "Full connectivity"],
    errorbar=("ci", 95),
    gap=0.1,
    width=0.65,
    ax=axes[0],
)

plot_data = pl.DataFrame(
    {
        "Improvement": np.concat((diffs_train, diffs_eval)),
        "Split": ["Calibration"] * diffs_train.size + ["Validation"] * diffs_eval.size,
    }
)

sns.barplot(
    plot_data,
    hue="Split",
    y="Improvement",
    errorbar=("sd", 1.96),
    gap=0.2,
    palette=["tab:blue", "tab:orange"],
    legend=False,
    ax=axes[1],
)
axes[1].set_xticks([-0.2, 0.2], ["Calibration", "Validation"])

for ax in axes:
    ax.set_xlabel("")
    ax.spines.top.set_visible(False)
    ax.spines.right.set_visible(False)

axes[0].set_ylabel("Mean relative entropy\n (bits/spin)")
axes[1].set_ylabel("Mean difference in\nrelative entropy")

axes[0].legend(
    loc="lower center",
    bbox_to_anchor=(0.5, 1.0),
    ncols=2,
    fontsize=8,
    title=None,
    frameon=False,
)

In [ ]:
sns.barplot(plot_data, x="Split", y="Improvement", errorbar=("sd", 1.96))